%md
# TP Final

## Notebook 2: Entrenamiento y registro de experimentos con MLflow

**Objetivo:** entrenar varios modelos de clasificación sobre el mismo conjunto de datos y partición, registrando en MLflow los parámetros, las métricas y los artefactos de cada corrida para poder compararlos en el notebook 3.

**Modelos:** regresión logística (con estandarización), random forest y LightGBM, cada uno con distintas configuraciones de hiperparámetros.

**Métricas:** accuracy, precision, recall, F1 y ROC-AUC sobre el conjunto de prueba, más el ROC-AUC de entrenamiento para detectar sobreajuste.

In [0]:
%pip install lightgbm

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import lightgbm
print(lightgbm.__version__)

4.7.0


##Paso 1: Carga de datos y partición

Se repite la carga, la definición de `bueno` y la partición 80/20 del notebook 1, con la misma semilla (`random_state=42`), para que todos los modelos usen exactamente los mismos conjuntos.

In [0]:
df = (spark.read.csv("dbfs:/databricks-datasets/wine-quality/winequality-red.csv",
                     header=True, sep=";", inferSchema=True)).toPandas()
df["bueno"] = (df["quality"] >= 6).astype(int)

from sklearn.model_selection import train_test_split
X = df.drop(columns=["quality", "bueno"]); y = df["bueno"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)   

(1279, 11) (320, 11)



##Paso 2: Configuración del experimento de MLflow

Se crea un experimento (`tp_final_vino`) donde quedan registradas todas las corridas. Cada corrida guarda parámetros, métricas, el modelo con su firma y la matriz de confusión.

In [0]:
import mlflow, mlflow.sklearn
import matplotlib.pyplot as plt
from mlflow.models import infer_signature

usuario = spark.sql("select current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/tp_final_vino")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1823981784236569', creation_time=1793232317052, experiment_id='1823981784236569', last_update_time=1793232492706, lifecycle_stage='active', name='/Users/egomezcoumroglon@itba.edu.ar/tp_final_vino', tags={'mlflow.experiment.sourceName': '/Users/egomezcoumroglon@itba.edu.ar/tp_final_vino',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'egomezcoumroglon@itba.edu.ar',
 'mlflow.ownerId': '79105209073163'}, trace_location=None, workspace='default'>

##Paso 3: Función de entrenamiento y registro
Se define una función que entrena un modelo, calcula las métricas y registra en MLflow los parámetros, las métricas, la matriz de confusión y el modelo con su firma. Así todas las corridas se registran de la misma forma y son comparables.

In [0]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay)

def correr(nombre, familia, modelo, params):
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({"modelo": familia, **params, "test_size": 0.2, "seed": 42})
        mlflow.set_tag("dataset", "wine-quality-red")
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_test)
        prob = modelo.predict_proba(X_test)[:, 1]
        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, prob),
            "train_roc_auc": roc_auc_score(y_train, modelo.predict_proba(X_train)[:, 1])})
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax)
        mlflow.log_figure(fig, "matriz_confusion.png"); plt.close(fig)
        mlflow.sklearn.log_model(modelo, artifact_path="model",
            signature=infer_signature(X_train, modelo.predict(X_train)))
        print(nombre, "| roc_auc:", round(roc_auc_score(y_test, prob), 3),
              "| f1:", round(f1_score(y_test, pred), 3))

##Paso 4. Corrida de prueba

Antes de lanzar los barridos, se entrena un único modelo (regresión logística con `C=1`) para verificar que el registro en MLflow funciona correctamente.

In [0]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

correr("logistica_C1", "logistica",
       make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=1000)), {"C": 1})

2026/10/30 03:04:17 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/10/30 03:04:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:04:19 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-0fa6d2ce8a5e4d2c8673f6ad65004bd0?o=7474650610073153


logistica_C1 | roc_auc: 0.824 | f1: 0.752


##Paso 5. Barridos de hiperparámetros
Se entrenan tres familias de modelos con distintas configuraciones: regresión logística (variando la regularización `C`), random forest (variando cantidad de árboles y profundidad) y LightGBM (variando tasa de aprendizaje, número de hojas y cantidad de árboles).

In [0]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

for C in [0.1, 10]:   # C=1 ya se corrió en la celda anterior
    correr(f"logistica_C{C}", "logistica",
           make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000)), {"C": C})

for n, d in [(100, 5), (300, 10), (300, None)]:
    correr(f"rf_n{n}_d{d}", "random_forest",
           RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42),
           {"n_estimators": n, "max_depth": str(d)})

for lr, leaves, n in [(0.1, 15, 100), (0.05, 31, 300), (0.05, 15, 300)]:
    correr(f"lgbm_lr{lr}_l{leaves}_n{n}", "lightgbm",
           LGBMClassifier(learning_rate=lr, num_leaves=leaves, n_estimators=n,
                          random_state=42, verbose=-1),
           {"learning_rate": lr, "num_leaves": leaves, "n_estimators": n})

2026/10/30 03:04:25 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/10/30 03:04:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:04:27 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-e755ca0e902b42088487c8319d079760?o=7474650610073153
2026/10/30 03:04:31 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


logistica_C0.1 | roc_auc: 0.825 | f1: 0.749


2026/10/30 03:04:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:04:32 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-01a3506880eb4e5d8ac9d06053a30187?o=7474650610073153
2026/10/30 03:04:37 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


logistica_C10 | roc_auc: 0.824 | f1: 0.754


2026/10/30 03:04:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:04:38 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-b73ff43dbeae4a41917c49bf0ced8e13?o=7474650610073153
2026/10/30 03:04:42 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


rf_n100_d5 | roc_auc: 0.839 | f1: 0.77


... (6 outputs left out)

2026/10/30 03:05:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:05:02 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-739b4270510249e5b9e5db45e8a53747?o=7474650610073153
2026/10/30 03:05:06 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


lgbm_lr0.05_l31_n300 | roc_auc: 0.896 | f1: 0.826


2026/10/30 03:05:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 03:05:07 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-15d6b6b5-5377.cloud.databricks.com/ml/experiments/1823981784236569/models/m-f485b642b4aa4ea79073e7c6f59cae80?o=7474650610073153


lgbm_lr0.05_l15_n300 | roc_auc: 0.883 | f1: 0.819


##Paso 6: Resultado
Se registraron 9 corridas en el experimento `tp_final_vino` (3 de regresión logística, 3 de random forest y 3 de LightGBM). La comparación y la selección del mejor modelo se hacen en el notebook 3.